In [1]:
# --- ESCUDO PARA EL ERROR DE LA LÍNEA 1 ---
import cv2
import numpy as np
import time
from IPython.display import display, Javascript, clear_output
from google.colab.output import eval_js
from base64 import b64decode
from google.colab.patches import cv2_imshow

# ==========================================
# 1. INICIALIZAR CONTADORES (Memoria)
# ==========================================
if 'total_buenos' not in globals():
    total_buenos = 0
if 'total_malos' not in globals():
    total_malos = 0

# ==========================================
# 2. FUNCIÓN PARA FOTO AUTOMÁTICA (CADA 5 SEG)
# ==========================================
def take_photo_auto(filename='photo.jpg', quality=0.8):
    js = Javascript('''
        async function takePhotoAuto(quality) {
            const div = document.createElement('div');

            // Texto indicador de tiempo
            const info = document.createElement('h3');
            info.style.color = '#ff9800';
            info.style.fontFamily = 'sans-serif';
            info.textContent = '⏱️ Cámara activa: Tomando foto automáticamente en 5 segundos...';
            div.appendChild(info);

            const video = document.createElement('video');
            video.style.display = 'block';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();

            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

            return new Promise((resolve) => {
                // Esperar exactamente 5000 milisegundos (5 segundos) antes de tomar la foto
                setTimeout(() => {
                    const canvas = document.createElement('canvas');
                    canvas.width = video.videoWidth;
                    canvas.height = video.videoHeight;
                    canvas.getContext('2d').drawImage(video, 0, 0);
                    stream.getVideoTracks()[0].stop();
                    div.remove();
                    resolve(canvas.toDataURL('image/jpeg', quality));
                }, 5000);
            });
        }
        ''')
    display(js)
    data = eval_js('takePhotoAuto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

# ==========================================
# 3. PROCESAMIENTO Y LÓGICA DE LEDs
# ==========================================
def procesar_imagen_y_contar(imagen):
    hsv = cv2.cvtColor(imagen, cv2.COLOR_BGR2HSV)

    # --- RANGOS DE COLORES ---
    lower_green = np.array([35, 80, 50])
    upper_green = np.array([85, 255, 255])

    lower_yellow = np.array([15, 80, 50])
    upper_yellow = np.array([34, 255, 255])

    lower_black = np.array([0, 0, 0])
    upper_black = np.array([180, 255, 60])

    # --- MÁSCARAS ---
    mask_green = cv2.inRange(hsv, lower_green, upper_green)
    mask_bad = cv2.bitwise_or(
        cv2.inRange(hsv, lower_yellow, upper_yellow),
        cv2.inRange(hsv, lower_black, upper_black)
    )

    buenos_en_foto = 0
    malos_en_foto = 0

    contornos_buenos, _ = cv2.findContours(mask_green, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in contornos_buenos:
        if cv2.contourArea(c) > 500:
            buenos_en_foto += 1
            x, y, w, h = cv2.boundingRect(c)
            cv2.rectangle(imagen, (x, y), (x+w, y+h), (0, 255, 0), 3)
            cv2.putText(imagen, 'BUENO', (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    contornos_malos, _ = cv2.findContours(mask_bad, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in contornos_malos:
        if cv2.contourArea(c) > 500:
            malos_en_foto += 1
            x, y, w, h = cv2.boundingRect(c)
            cv2.rectangle(imagen, (x, y), (x+w, y+h), (0, 0, 255), 3)
            cv2.putText(imagen, 'MALO', (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    # Ajustar tamaño para la pantalla
    alto, ancho = imagen.shape[:2]
    if ancho > 600:
        imagen = cv2.resize(imagen, (600, int(alto * (600/ancho))))

    cv2_imshow(imagen)
    return buenos_en_foto, malos_en_foto

# ==========================================
# 4. BUCLE INFINITO (CADA 5 SEG)
# ==========================================
try:
    while True:
        clear_output(wait=True) # Limpia la pantalla de Colab para que no se haga una lista infinita hacia abajo
        print("========================================")
        print(f"📊 CONTEO ACUMULADO: {total_buenos} Buenos | {total_malos} Malos")
        print("========================================")
        print("Presiona el botón de 'Stop' ⬛ en Colab para detener el ciclo.\n")

        # 1. Abre la cámara, espera 5 segs y toma la foto
        photo_path = take_photo_auto()

        # 2. Lee la foto recién tomada
        img_camera = cv2.imread(photo_path)

        if img_camera is not None:
            # 3. Analiza la foto
            b, m = procesar_imagen_y_contar(img_camera)

            # 4. Suma a la memoria global
            total_buenos += b
            total_malos += m

            print("\n--- RESULTADO DE ESTA FOTO ---")
            if b > 0:
                print(f"🟢 PIN VERDE: ON ({b} limón verde detectado)")
            if m > 0:
                print(f"🔴 PIN ROJO: ON ({m} limón malo detectado)")
            if b == 0 and m == 0:
                print("⚪ PINES APAGADOS (Solo se ve la banda gris)")

            print(f"\n✅ Total Acumulado - BUENOS: {total_buenos} | MALOS: {total_malos}")

            # Pequeña pausa antes de abrir la cámara de nuevo para no saturar Colab
            time.sleep(1)

        else:
            print("Error al leer la imagen capturada.")

except KeyboardInterrupt:
    print("\n========================================")
    print("🛑 CICLO DETENIDO POR EL USUARIO")
    print(f"TOTAL FINAL -> BUENOS: {total_buenos} | MALOS: {total_malos}")
    print("========================================")
except Exception as err:
    print(f"\nError: {str(err)}")

📊 CONTEO ACUMULADO: 14 Buenos | 14 Malos
Presiona el botón de 'Stop' ⬛ en Colab para detener el ciclo.



<IPython.core.display.Javascript object>


🛑 CICLO DETENIDO POR EL USUARIO
TOTAL FINAL -> BUENOS: 14 | MALOS: 14
